In [ ]:
#Checking if GPU is running or not
!nvidia-smi

In [ ]:
!pip install datasets transformers[sentencepiece] sacrebleu -q

In [ ]:
import sys
import transformers
import tensorflow as tf
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

In [ ]:
model_checkpoint = "Helsinki-NLP/opus-mt-en-hi"

In [ ]:
raw_datasets = load_dataset("cfilt/iitb-english-hindi")

In [ ]:
raw_datasets

In [ ]:
raw_datasets['train'][0]

**Preprocessing the** **data**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
tokenizer("Hello, this is a sentencs!")

In [ ]:
tokenizer(["Hello, this is a sentence!", "This is another sentence."])

In [ ]:
print(tokenizer(text_target=["Hello,this is a sentence", "This is another sentence."]))

In [ ]:
max_input_length=128
max_target_length=128

source_lang="en"
target_lang="hi"

def preprocess_function(examples):
    inputs = [x["en"] for x in examples["translation"]]
    targets = [x["hi"] for x in examples["translation"]]

    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
print(raw_datasets["train"].column_names)


In [ ]:
print(raw_datasets["train"][:2])

In [ ]:
preprocess_function(raw_datasets["train"][:2])

In [ ]:
tokenized_datasets=raw_datasets.map(preprocess_function,batched=True)

In [ ]:
model=AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

In [ ]:
batch_size=16
learning_rate=2e-5
weight_decay=0.01
num_train_epochs=5

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    return_tensors="pt"
)

In [ ]:
generation_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    return_tensors="pt"
)

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

In [ ]:
small_train_dataset = tokenized_datasets["train"].select(range(5000))
small_val_dataset = tokenized_datasets["validation"].select(range(520))

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=True,
    logging_steps=100
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

In [ ]:
trainer.train()


In [ ]:
generation_dataset = tokenized_datasets["validation"]

**Model Testing**

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

text = "How are you?"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
text = "How are you?"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
text = "My name is Sandhya"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
text = "What are you doing?"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
text = "Why are you laughing?"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))